# 02 — Clinical Data Exploration

Loads the DAP dataset and the Kaggle children's drawings dataset,
visualises samples, reports image-size and format distributions,
checks deviation-label balance, and flags data-quality issues.

In [2]:
import os, sys, pathlib, zipfile, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import seaborn as sns

sys.path.insert(0, os.path.join('..', 'src'))
print('Imports OK')

Imports OK


In [ ]:
import json, pathlib

# ── Paste your Kaggle credentials here ────────────────────────────────────
KAGGLE_USERNAME = 'jayanthsibbala'
KAGGLE_KEY      = 'KGAT_804044901b8dcd686ffedd138f9ecae5'
# ──────────────────────────────────────────────────────────────────────────

kaggle_dir = pathlib.Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
creds_path = kaggle_dir / 'kaggle.json'
creds_path.write_text(json.dumps({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}))
creds_path.chmod(0o600)

print(f'kaggle.json written to {creds_path}')

In [ ]:
import kaggle

DAP_DIR      = pathlib.Path('../data/raw/dap')
CHILDREN_DIR = pathlib.Path('../data/raw/kaggle_children')
DAP_DIR.mkdir(parents=True, exist_ok=True)
CHILDREN_DIR.mkdir(parents=True, exist_ok=True)

kaggle.api.authenticate()
print('Authenticated with Kaggle.')

def kaggle_download(dataset: str, dest: pathlib.Path):
    if any(dest.iterdir()):
        print(f'[SKIP] {dest} already has files.')
        return
    print(f'Downloading {dataset} → {dest} ...')
    kaggle.api.dataset_download_files(dataset, path=str(dest), unzip=True, quiet=False)
    files = list(dest.rglob('*'))
    print(f'Done. {len(files)} files in {dest}')

kaggle_download('lachin007/drawaperson-handdrawn-sketches-by-children', DAP_DIR)
kaggle_download('vishmiperera/children-drawings', CHILDREN_DIR)

## 1b. C-OBGET Dataset (Mendeley Data)

The **C-OBGET** dataset contains 386 Bender-Gestalt Test drawings from children aged 4–11,
labeled by a psychologist across 15 disorder categories (ADHD, Depression, Anxiety, Learning Disorders, Normal, etc.)
and 9 per-figure pattern labels.

**Manual download required (one time):**
1. Go to: https://data.mendeley.com/datasets/62kwttmkcv/1
2. Click **Download All** — saves a zip file (~some MB)
3. Move the zip to: `drawnet/data/raw/obget/`
4. Run the cell below — it will unzip and report the structure automatically

In [29]:
import zipfile, collections

OBGET_DIR = pathlib.Path('../data/raw/obget')
OBGET_DIR.mkdir(parents=True, exist_ok=True)

# Find the downloaded zip
zips = list(OBGET_DIR.glob('*.zip'))

if not zips:
    print('No zip found in data/raw/obget/')
    print('Please download from: https://data.mendeley.com/datasets/62kwttmkcv/1')
    print('Then move the zip into drawnet/data/raw/obget/ and re-run this cell.')
else:
    zip_path = zips[0]
    print(f'Found: {zip_path.name}  ({zip_path.stat().st_size / 1e6:.1f} MB)')

    # Unzip if not already done
    already_extracted = [p for p in OBGET_DIR.iterdir() if p.is_dir()]
    if not already_extracted:
        print('Extracting...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(OBGET_DIR)
        print('Done.')
    else:
        print('Already extracted.')

    # Report structure
    print('\n--- Folder structure ---')
    for p in sorted(OBGET_DIR.rglob('*')):
        if p.is_dir():
            n_files = len(list(p.glob('*.*')))
            depth   = len(p.relative_to(OBGET_DIR).parts)
            print('  ' * depth + f'{p.name}/  ({n_files} files)')

    # Count images per disorder label (assumes subfolders = labels)
    print('\n--- Label distribution ---')
    label_counts = collections.Counter()
    for f in OBGET_DIR.rglob('*'):
        if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
            label_counts[f.parent.name] += 1
    for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
        print(f'  {label:35s}: {count}')

Found: C-OBGET (OBGET for Children).zip  (34.0 MB)
Extracting...
Done.

--- Folder structure ---
  C-OBGET (OBGET for Children)/  (4 files)

--- Label distribution ---
  C-OBGET (OBGET for Children)       : 1


## 2. Inventory — file formats and counts

In [ ]:
def inventory_dir(root: pathlib.Path):
    exts = collections.Counter()
    all_files = []
    for p in root.rglob('*'):
        if p.is_file():
            exts[p.suffix.lower()] += 1
            all_files.append(p)
    return all_files, exts

dap_files,      dap_exts      = inventory_dir(DAP_DIR)
children_files, children_exts = inventory_dir(CHILDREN_DIR)

print(f'DAP dataset      — {len(dap_files)} files')
print('  Extensions:', dict(dap_exts))
print()
print(f'Children dataset — {len(children_files)} files')
print('  Extensions:', dict(children_exts))

## 3. Visualise sample images

In [ ]:
import random

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

def show_samples(files, title, n=16, ncols=4):
    img_files = [f for f in files if f.suffix.lower() in IMG_EXTS]
    if not img_files:
        print(f'No image files found for {title}')
        return
    sample = random.sample(img_files, min(n, len(img_files)))
    nrows = (len(sample) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3*nrows))
    axes = axes.flatten()
    fig.suptitle(title, fontsize=13)
    for ax, p in zip(axes, sample):
        try:
            img = Image.open(p).convert('RGB')
            ax.imshow(img)
            ax.set_title(p.parent.name, fontsize=7)
        except Exception as e:
            ax.set_title(f'ERROR: {e}', fontsize=6)
        ax.axis('off')
    for ax in axes[len(sample):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_samples(dap_files,      'DAP Dataset — Sample Images')
show_samples(children_files, 'Children Drawings — Sample Images')

## 4. Image size distribution

In [1]:
def collect_sizes(files):
    widths, heights, corrupt = [], [], []
    for f in files:
        if f.suffix.lower() not in IMG_EXTS:
            continue
        try:
            w, h = Image.open(f).size
            widths.append(w)
            heights.append(h)
        except Exception:
            corrupt.append(f)
    return widths, heights, corrupt

dap_w, dap_h, dap_corrupt             = collect_sizes(dap_files)
children_w, children_h, ch_corrupt    = collect_sizes(children_files)

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for ax, data, label in zip(axes.flatten(),
                            [dap_w, dap_h, children_w, children_h],
                            ['DAP Widths', 'DAP Heights',
                             'Children Widths', 'Children Heights']):
    ax.hist(data, bins=30, edgecolor='black', color='steelblue')
    ax.set_title(label)
    ax.set_xlabel('Pixels')
plt.tight_layout()
plt.show()

print(f'Corrupt / unreadable files — DAP: {len(dap_corrupt)}, Children: {len(ch_corrupt)}')
if dap_corrupt:
    print('  DAP corrupt:', [str(f) for f in dap_corrupt[:5]])
if ch_corrupt:
    print('  Children corrupt:', [str(f) for f in ch_corrupt[:5]])

NameError: name 'dap_files' is not defined

## 5. Class / label distribution

Classes are inferred from subdirectory names. If the datasets ship with
a CSV annotation file, update the path below.

In [ ]:
def class_counts_from_dirs(files):
    counts = collections.Counter()
    for f in files:
        if f.suffix.lower() in IMG_EXTS:
            counts[f.parent.name] += 1
    return counts

dap_classes      = class_counts_from_dirs(dap_files)
children_classes = class_counts_from_dirs(children_files)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, counts, title in zip(axes,
                              [dap_classes, children_classes],
                              ['DAP — Class Distribution',
                               'Children Drawings — Class Distribution']):
    if counts:
        labels, vals = zip(*sorted(counts.items(), key=lambda x: -x[1]))
        ax.bar(labels, vals, edgecolor='black', color='coral')
        ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_title(title)
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 6. Deviation label distribution (placeholder)

Update `ANNOTATION_CSV` with the actual path once you have annotated or
sourced ground-truth deviation labels.

In [ ]:
DEVIATION_CLASSES = [
    'rotation', 'closure_failure', 'perseveration',
    'spatial_disorganization', 'size_distortion', 'omission'
]

ANNOTATION_CSV = '../data/processed/deviation/annotations.csv'

if os.path.exists(ANNOTATION_CSV):
    df = pd.read_csv(ANNOTATION_CSV)
    dev_counts = df[DEVIATION_CLASSES].sum()

    plt.figure(figsize=(8, 4))
    dev_counts.plot(kind='bar', edgecolor='black', color='mediumpurple')
    plt.title('Deviation Label Frequency')
    plt.ylabel('Positive samples')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

    print('Imbalance ratios (positive / total):')
    print((dev_counts / len(df)).round(3).to_string())
else:
    print(f'Annotation CSV not found at {ANNOTATION_CSV}.')
    print('Deviation label analysis will be available once annotations are created.')
    print('Expected columns:', ['filepath'] + DEVIATION_CLASSES)

## 7. Data quality summary

In [ ]:
print('=' * 60)
print('DATA QUALITY SUMMARY')
print('=' * 60)

print(f'\nDAP dataset')
print(f'  Total image files : {len(dap_w)}')
print(f'  Corrupt files     : {len(dap_corrupt)}')
print(f'  Classes found     : {len(dap_classes)}')
if dap_w:
    print(f'  Width  range      : {min(dap_w)} – {max(dap_w)} px')
    print(f'  Height range      : {min(dap_h)} – {max(dap_h)} px')

print(f'\nChildren drawings')
print(f'  Total image files : {len(children_w)}')
print(f'  Corrupt files     : {len(ch_corrupt)}')
print(f'  Classes found     : {len(children_classes)}')
if children_w:
    print(f'  Width  range      : {min(children_w)} – {max(children_w)} px')
    print(f'  Height range      : {min(children_h)} – {max(children_h)} px')

print()
print('Action items:')
print('  1. Remove/repair corrupt files listed above.')
print('  2. Resize all images to 224x224 before training.')
print('  3. Create deviation annotations CSV (see cell 6).')
print('  4. Address class imbalance — consider weighted sampling or focal loss.')